In [ ]:
"""
NVIDIA Nemotron Reasoning Challenge â€” Training Notebook v4
Strategy: Fine-tune LoRA adapter with chain-of-thought reasoning data
"""

In [ ]:
import subprocess, sys, os

# Install bitsandbytes (not available by default in Kaggle nvidia-utility-script env)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.46.1"])

# Check GPU compute capability
import torch
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name} (CC {cc[0]}.{cc[1]})")
    if cc[0] < 7:
        print("P100 detected â€” reinstalling PyTorch with CUDA 12.1 (supports sm_60)...")
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
            "torch", "torchvision", "torchaudio", "causal-conv1d", "mamba-ssm"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
            "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
            "--no-cache-dir", "mamba-ssm[causal-conv1d]"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.46.1"])
        print("PyTorch reinstalled for P100 compatibility")
        # Re-import torch
        import importlib
        importlib.invalidate_caches()
        import torch
        print(f"New torch version: {torch.__version__}, CUDA: {torch.version.cuda}")

In [ ]:
import os
import gc
import re
import json
import zipfile
import random
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Try multiple possible data paths
for base in [
    "/kaggle/input/nvidia-nemotron-model-reasoning-challenge",
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge",
    "/kaggle/input/nvidia-nemotron-3-reasoning-challenge",
]:
    if os.path.exists(os.path.join(base, "train.csv")):
        TRAIN_PATH = os.path.join(base, "train.csv")
        TEST_PATH = os.path.join(base, "test.csv")
        print(f"Found data at: {base}")
        break
else:
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))
    raise FileNotFoundError("Could not find competition data!")

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
print(f"Train samples: {len(train_df)}")

In [ ]:
VOCAB = {
    'above', 'alice', 'ancient', 'around', 'beyond', 'bird', 'book', 'bright',
    'castle', 'cat', 'cave', 'chases', 'clever', 'colorful', 'creates', 'crystal',
    'curious', 'dark', 'discovers', 'door', 'dragon', 'draws', 'dreams', 'explores',
    'follows', 'forest', 'found', 'garden', 'golden', 'hatter', 'hidden', 'imagines',
    'in', 'inside', 'island', 'key', 'king', 'knight', 'library', 'magical', 'map',
    'message', 'mirror', 'mountain', 'mouse', 'mysterious', 'near', 'ocean', 'palace',
    'potion', 'princess', 'puzzle', 'queen', 'rabbit', 'reads', 'school', 'secret',
    'sees', 'silver', 'story', 'strange', 'student', 'studies', 'teacher', 'the',
    'through', 'tower', 'treasure', 'turtle', 'under', 'valley', 'village', 'watches',
    'wise', 'wizard', 'wonderland', 'writes'
}
VOCAB_BY_LEN = {}
for w in VOCAB:
    VOCAB_BY_LEN.setdefault(len(w), []).append(w)


def categorize(prompt):
    if 'gravitational constant' in prompt: return 'gravity'
    if 'unit conversion' in prompt: return 'unit_conversion'
    if 'numbers are secretly converted' in prompt: return 'number_conversion'
    if 'encryption rules' in prompt: return 'encryption'
    if 'bit manipulation' in prompt: return 'bit_manipulation'
    if 'transformation rules' in prompt: return 'symbol_transform'
    return 'unknown'


def solve_gravity_with_cot(prompt):
    examples = re.findall(r'For t = ([\d.]+)s, distance = ([\d.]+) m', prompt)
    query = re.search(r'for t = ([\d.]+)s given', prompt)
    if not query: return None, ""

    t_q = float(query.group(1))
    ts = [float(e[0]) for e in examples]
    ds = [float(e[1]) for e in examples]
    ds_str = [e[1] for e in examples]

    g_values = [2*d/(t**2) for t, d in zip(ts, ds)]
    ts_np, ds_np = np.array(ts), np.array(ds)
    g_lsq = float(2 * np.sum(ds_np * ts_np**2) / np.sum(ts_np**4))

    candidates = g_values + [g_lsq, float(np.median(g_values))]

    best_g = candidates[0]
    best_score = -1
    best_residual = float('inf')
    for g in candidates:
        score = sum(1 for t, d_s in zip(ts, ds_str) if f"{0.5*g*t**2:.2f}" == d_s)
        residual = sum(abs(0.5*g*t**2 - d) for t, d in zip(ts, ds))
        if score > best_score or (score == best_score and residual < best_residual):
            best_score, best_residual, best_g = score, residual, g

    answer = f"{0.5 * best_g * t_q**2:.2f}"
    cot = (f"Using d = 0.5*g*t^2, I compute g from the examples.\n"
           f"From example 1: g = 2*{ds[0]}/{ts[0]}^2 = {g_values[0]:.4f}\n"
           f"Best g = {best_g:.4f} (matches {best_score}/{len(ts)} examples).\n"
           f"For t = {t_q}: d = 0.5 * {best_g:.4f} * {t_q}^2 = {answer}")
    return answer, cot


def solve_unit_with_cot(prompt):
    examples = re.findall(r'([\d.]+) m becomes ([\d.]+)', prompt)
    query = re.search(r'convert the following measurement: ([\d.]+) m', prompt)
    if not query: return None, ""

    x_q = float(query.group(1))
    xs = [float(e[0]) for e in examples]
    ys = [float(e[1]) for e in examples]
    ys_str = [e[1] for e in examples]
    factors = [y/x for x, y in zip(xs, ys)]

    xs_np, ys_np = np.array(xs), np.array(ys)
    f_lsq = float(np.sum(xs_np*ys_np) / np.sum(xs_np**2))

    candidates = factors + [f_lsq, float(np.median(factors))]

    best_f = candidates[0]
    best_score = -1
    best_residual = float('inf')
    for f in candidates:
        score = sum(1 for x, y_s in zip(xs, ys_str) if f"{f*x:.2f}" == y_s)
        residual = sum(abs(f*x - y) for x, y in zip(xs, ys))
        if score > best_score or (score == best_score and residual < best_residual):
            best_score, best_residual, best_f = score, residual, f

    answer = f"{best_f * x_q:.2f}"
    cot = (f"The conversion factor = output/input.\n"
           f"From example 1: {ys[0]}/{xs[0]} = {factors[0]:.4f}\n"
           f"Best factor = {best_f:.4f} (matches {best_score}/{len(xs)} examples).\n"
           f"For {x_q} m: result = {best_f:.4f} * {x_q} = {answer}")
    return answer, cot


def int_to_roman(num):
    val = [1000,900,500,400,100,90,50,40,10,9,5,4,1]
    syms = ['M','CM','D','CD','C','XC','L','XL','X','IX','V','IV','I']
    result = ''
    for i in range(len(val)):
        while num >= val[i]:
            result += syms[i]
            num -= val[i]
    return result


def solve_number_with_cot(prompt):
    query = re.search(r'write the number (\d+) in the Wonderland', prompt)
    if not query: return None, ""
    num = int(query.group(1))
    answer = int_to_roman(num)

    roman_map = [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),(100,'C'),(90,'XC'),
                 (50,'L'),(40,'XL'),(10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
    steps = []
    remainder = num
    for v, s in roman_map:
        if remainder >= v:
            count = remainder // v
            steps.append(f"{remainder} >= {v}: add {''.join([s]*count)}")
            remainder -= v * count

    cot = (f"The examples show decimal to Roman numeral conversion.\n"
           f"Converting {num}:\n" + "\n".join(steps) + f"\nResult: {answer}")
    return answer, cot


def solve_encryption_with_cot(prompt):
    parts = prompt.split('Now, decrypt the following text:')
    if len(parts) != 2: return None, ""
    query = parts[1].strip()

    char_map = {}
    for line in parts[0].strip().split('\n'):
        if ' -> ' not in line: continue
        encrypted, decrypted = line.split(' -> ', 1)
        enc_words, dec_words = encrypted.strip().split(), decrypted.strip().split()
        if len(enc_words) != len(dec_words): continue
        for ew, dw in zip(enc_words, dec_words):
            if len(ew) != len(dw): continue
            for ec, dc in zip(ew, dw):
                if ec.isalpha() and dc.isalpha():
                    char_map[ec] = dc

    result_words = []
    for qw in query.split():
        decrypted_chars = []
        missing_positions = []
        for i, c in enumerate(qw):
            if c in char_map:
                decrypted_chars.append(char_map[c])
            else:
                decrypted_chars.append(None)
                missing_positions.append(i)

        if not missing_positions:
            result_words.append(''.join(decrypted_chars))
        else:
            candidates_list = VOCAB_BY_LEN.get(len(qw), [])
            best_match = None
            for vocab_word in candidates_list:
                match = True
                new_maps = {}
                conflict = False
                for i, (dc, vc) in enumerate(zip(decrypted_chars, vocab_word)):
                    if dc is not None and dc != vc:
                        match = False; break
                if not match: continue
                for i in missing_positions:
                    enc_c, dec_c = qw[i], vocab_word[i]
                    if enc_c in char_map and char_map[enc_c] != dec_c:
                        conflict = True; break
                    if enc_c in new_maps and new_maps[enc_c] != dec_c:
                        conflict = True; break
                    for k, v in char_map.items():
                        if v == dec_c and k != enc_c:
                            conflict = True; break
                    if conflict: break
                    new_maps[enc_c] = dec_c
                if not conflict:
                    best_match = vocab_word
                    char_map.update(new_maps)
                    break
            result_words.append(best_match if best_match else ''.join(c if c else '?' for c in decrypted_chars))

    answer = ' '.join(result_words)
    map_sample = list(char_map.items())[:5]
    cot = (f"This is a substitution cipher. Building decryption map from examples:\n"
           f"Mappings: {', '.join(f'{k}->{v}' for k,v in map_sample)}...\n"
           f"({len(char_map)} letters mapped)\n"
           f"Decrypting '{query[:50]}...' gives: {answer}")
    return answer, cot


def solve_bit_with_cot(prompt, answer_gt):
    """Try to identify bit manipulation pattern, use ground truth."""
    pairs = re.findall(r'([01]{8}) -> ([01]{8})', prompt)
    query_m = re.search(r'determine the output for: ([01]{8})', prompt)
    if not pairs or not query_m:
        return answer_gt, f"Apply the bit transformation rule: {answer_gt}"

    query = query_m.group(1)

    # Try XOR with constant
    first_xor = int(pairs[0][0], 2) ^ int(pairs[0][1], 2)
    if all(int(a, 2) ^ first_xor == int(b, 2) for a, b in pairs):
        result = format(int(query, 2) ^ first_xor, '08b')
        cot = (f"Checking XOR pattern:\n"
               f"  {pairs[0][0]} XOR mask = {pairs[0][1]}\n"
               f"  mask = {first_xor:08b}\n"
               f"All examples match XOR with {first_xor:08b}.\n"
               f"Applying: {query} XOR {first_xor:08b} = {result}")
        return result, cot

    # Try NOT (complement)
    if all(int(a, 2) ^ 0xFF == int(b, 2) for a, b in pairs):
        result = format(int(query, 2) ^ 0xFF, '08b')
        cot = f"Each bit is flipped (NOT operation).\n{query} -> {result}"
        return result, cot

    # Fallback: use ground truth with descriptive CoT
    cot = (f"Analyzing bit transformation from {len(pairs)} examples:\n"
           f"  {pairs[0][0]} -> {pairs[0][1]}\n"
           f"  {pairs[1][0]} -> {pairs[1][1]}\n"
           f"The pattern involves a complex combination of bit operations.\n"
           f"Applying the discovered rule to {query}: {answer_gt}")
    return answer_gt, cot


def solve_with_cot(prompt, answer_ground_truth):
    """Returns (answer, chain_of_thought). Uses ground truth as fallback."""
    cat = categorize(prompt)

    if cat == 'gravity':
        ans, cot = solve_gravity_with_cot(prompt)
    elif cat == 'unit_conversion':
        ans, cot = solve_unit_with_cot(prompt)
    elif cat == 'number_conversion':
        ans, cot = solve_number_with_cot(prompt)
    elif cat == 'encryption':
        ans, cot = solve_encryption_with_cot(prompt)
    elif cat == 'bit_manipulation':
        ans, cot = solve_bit_with_cot(prompt, str(answer_ground_truth))
        return ans, cot
    else:
        ans, cot = None, ""

    # Use ground truth if solver failed
    if ans is None or str(ans).strip() != str(answer_ground_truth).strip():
        ans = str(answer_ground_truth)
        if cat == 'symbol_transform':
            cot = (f"Analyzing the transformation pattern from examples.\n"
                   f"Each symbol follows a specific mapping rule.\n"
                   f"Applying the transformation: {ans}")
        else:
            cot = f"Working through the pattern: {ans}"

    return ans, cot

In [ ]:
print("Generating CoT training data...")
train_df['category'] = train_df['prompt'].apply(categorize)

training_texts = []
for idx, row in train_df.iterrows():
    answer, cot = solve_with_cot(row['prompt'], row['answer'])

    user_msg = row['prompt'] + "\nPlease reason step by step and put your final answer inside \\boxed{}."
    assistant_msg = f"<think>\n{cot}\n</think>\n\n\\boxed{{{answer}}}"

    training_texts.append({
        'prompt': user_msg,
        'response': assistant_msg,
        'category': row['category']
    })

cat_counts = train_df['category'].value_counts()
print(f"Category distribution:\n{cat_counts}")

# Oversample hard categories 2x
final_texts = training_texts.copy()
for text in training_texts:
    if text['category'] in ['bit_manipulation', 'symbol_transform']:
        final_texts.append(text)

random.shuffle(final_texts)
print(f"Total training examples: {len(final_texts)}")

In [ ]:
import shutil
import stat

# Fix Triton ptxas-blackwell permission issue
try:
    src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
    dst = "/tmp/ptxas-blackwell"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        import triton.backends.nvidia as nv_backend
        src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
        dst_bin = "/tmp/triton_nvidia_bin"
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f_name in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f_name)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
        os.environ["TRITON_PTXAS_PATH"] = dst
        print("Triton ptxas fix applied")
except Exception as e:
    print(f"Triton fix skipped: {e}")

import kagglehub
import mamba_ssm
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# Detect GPU
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
gpu_cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name} (CC {gpu_cc[0]}.{gpu_cc[1]}, {gpu_mem:.1f} GB)")

# Use 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if gpu_cc[0] >= 8 else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

os.makedirs("/tmp/offload", exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    offload_folder="/tmp/offload",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Bypass Mamba fast-path CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Vocab: {len(tokenizer)}")

# Prepare for quantized training
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA config
LORA_RANK = 32
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=64,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 1536
BATCH_SIZE = 1
GRAD_ACCUM = 8
NUM_EPOCHS = 2
LR = 2e-4

def build_chat_text(prompt, response):
    try:
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        text = (
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n{response}<|im_end|>"
        )
    return text

# Use all training data (no sampling)
TRAIN_SAMPLE = min(6000, len(final_texts))
sampled_texts = random.sample(final_texts, TRAIN_SAMPLE)

tokenized_texts = [
    build_chat_text(t['prompt'], t['response']) for t in sampled_texts
]
print(f"Prepared {len(tokenized_texts)} training examples")

class SFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = []
        for text in texts:
            enc = tokenizer(
                text, truncation=True, max_length=max_length,
                padding="max_length", return_tensors="pt"
            )
            ids = enc["input_ids"].squeeze(0)
            mask = enc["attention_mask"].squeeze(0)
            labels = ids.clone()
            labels[mask == 0] = -100
            self.encodings.append({"input_ids": ids, "attention_mask": mask, "labels": labels})

    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx): return self.encodings[idx]

dataset = SFTDataset(tokenized_texts, tokenizer, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Bypass Triton rmsnorm with pure PyTorch fallback
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

model.train()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01
)

from torch.optim.lr_scheduler import CosineAnnealingLR
total_steps = (len(dataloader) * NUM_EPOCHS) // GRAD_ACCUM
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR/10)

print(f"Training: {NUM_EPOCHS} epochs, ~{total_steps} optimizer steps, {len(dataset)} examples")

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()

    for i, batch in enumerate(dataloader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        running_loss += outputs.loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            step = (i + 1) // GRAD_ACCUM
            if step % 50 == 0:
                avg = running_loss / (i + 1)
                print(f"  epoch {epoch+1} | step {step}/{total_steps} | avg_loss {avg:.4f} | lr {scheduler.get_last_lr()[0]:.6f}")

    if (i + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} â€” avg loss: {avg_loss:.4f}")

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

for f_name in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f_name))
    print(f"  {f_name} ({size/1024:.1f} KB)")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("submission.zip verified OK")